# MVP: Análise de Dados e Boas Práticas - Pós-Graduação PUC-RIO
### Projeto apresentado como atividade final da Sprint Análise de Dados e Boas Práticas

**Autor:** Rogério dos Santos Ferreira  
**Matrícula:** 4052025001195  
**Data:** 04/04/2026

**Dataset:** [Airline Passenger Satisfaction](https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction)  
Quais fatores contribuem para a satisfação do cliente em uma companhia aérea?

<div style="page-break-after: always;"></div>  

# 1. Descrição do Problema

O objetivo deste projeto é a realização da Análise de Dados de um conjunto de informações acerca da satisfação geral de clientes em relação aos serviços prestados por companhia aérea, de forma a identificar possíveis correlações entre os atributos para a identificação de medidas mitigatórias. Além da análise citada os dados deverão ser preparados para o treinamento de um modelo de Machine Learning destinado à classificação dos clientes em graus de satisfação geral (Satisfeito, Neutro ou Insatisfeito), com base em seus atributos sócio-demográficos, no tipo de serviço contratado (classe) e na obtenção de respostas a questionamentos de satisfação específicos acerca da percepção de qualidade de detrminados serviços prestados.

## Hipóteses Levantadas

1. Verificar se ...

2. Existe correlação entre ...

3. O tratamento X é mais eficaz que o Y?

4. ...

## Tipo do Problema

O problema em questão trata-se da construção de um modelo de Machine Learning para a **Classificação Supervisionada** do grau de satisfação geral de clientes de uma companhia aérea com base em um conjunto de informações sócio-demográficas e resultados de consultas de satisfação específicas.

## Seleção de Dados

O Dataset utilizado foi obtido diretamente na plataforma Kaggle, não havendo necessidade de seleção de dados externos. O mesmo encontra-se disponível no endereço [Kaggle - Airline Passenger Satisfaction](https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction).

## Atributos do Dataset

O Dataset utilizado contém **129.880 amostras**, sedo as mesmas compostas de **23 atributos**, os quais são descritos a seguir:

**Gender**: Gênero dos passageiros (Feminino, Masculino)  
**Customer Type**: O tipo de cliente (Regular, Esporádico)  
**Age**: Idade atual dos passageiros  
**Type of Travel**: Objetivo do voo (Viagem Pessoal, Viagem de Negócios)  
**Class**: Classe da viagem (Executiva, Econômica, Econômica Plus)  
**Flight distance**: Distância de voo  
**Inflight wifi service**: Nível de satisfação com o serviço de Wi-Fi a bordo (0: Não aplicável; 1-5)  
**Departure/Arrival time convenient**: Nível de satisfação com a conveniência do horário de partida/chegada  
**Ease of Online booking**: Nível de satisfação com a facilidade da reserva online  
**Gate location**: Nível de satisfação com a localização do portão  
**Food and drink**: Nível de satisfação com comida e bebida  
**Online boarding**: Nível de satisfação com o check-in (embarque) online  
**Seat comfort**: Nível de satisfação com o conforto do assento  
**Inflight entertainment**: Nível de satisfação com o entretenimento de bordo  
**On-board service**: Nível de satisfação com o serviço de bordo  
**Leg room service**: Nível de satisfação com o espaço para as pernas  
**Baggage handling**: Nível de satisfação com o manuseio de bagagem  
**Check-in service**: Nível de satisfação com o serviço de check-in  
**Inflight service**: Nível de satisfação com o serviço de bordo (atendimento da tripulação)  
**Cleanliness**: Nível de satisfação com a limpeza  
**Departure Delay in Minutes**: Minutos de atraso no momento da partida  
**Arrival Delay in Minutes**: Minutos de atraso no momento da chegada  
**Satisfaction**: Nível de satisfação com a companhia aérea (Satisfeito, Neutro ou Insatisfeito)


<div style="page-break-after: always;"></div>  

# 2. Importação das Bibliotecas e Carga de Dados

Esta seção consolida todas as importações de bibliotecas necessárias para a análise, visualização e pré-processamento dos dados, bem como o carregamento inicial do dataset.

In [11]:
# Importação das bibliotecas necessárias para o projeto
import pandas as pd

from autogluon.tabular import TabularPredictor

# Carga dos dados de treinamento e teste
train_data = pd.read_csv(
    "../dados/originais/Airline Passenger Satisfaction - train.csv", index_col=0
)
test_data = pd.read_csv(
    "../dados/originais/Airline Passenger Satisfaction - test.csv", index_col=0
)

# Combinação dos dados de treinamento e teste para análise exploratória
data = pd.concat([train_data, test_data], ignore_index=True)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129880 entries, 0 to 129879
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 129880 non-null  int64  
 1   Gender                             129880 non-null  object 
 2   Customer Type                      129880 non-null  object 
 3   Age                                129880 non-null  int64  
 4   Type of Travel                     129880 non-null  object 
 5   Class                              129880 non-null  object 
 6   Flight Distance                    129880 non-null  int64  
 7   Inflight wifi service              129880 non-null  int64  
 8   Departure/Arrival time convenient  129880 non-null  int64  
 9   Ease of Online booking             129880 non-null  int64  
 10  Gate location                      129880 non-null  int64  
 11  Food and drink                     1298

In [15]:
# Treinamento de modelo de base para evolução posterior

# Definição da emente aleatória para reprodutibilidade dos resultados
RANDOM_SEED = 42

# Seleção dos modelos empregados
custom_hyperparameters = {
    "GBM": [{}],
    "CAT": [{}],
    "XGB": [{}],
}

# Treinamento do modelo de base utilizando AutoGluon
predictor = TabularPredictor(
    label="satisfaction",
    path="../modelos/",
    learner_kwargs={"random_state": RANDOM_SEED},
).fit(
    train_data,
    hyperparameters=custom_hyperparameters,
    presets="medium",
    time_limit=300,
    verbosity=0,
)

Preset alias specified: 'medium' maps to 'medium_quality'.


In [22]:
# Análise dos modelos empregados e identificação do melhor modelo
leaderboard = predictor.leaderboard(test_data, silent=True)
display(leaderboard)

# Análise de desempenho do melhor modelo
performance = predictor.evaluate(test_data)

print("Desempenho do melhor modelo:")
print(f"Acurácia: {round(performance['accuracy'], 4)}")
print(f"Precision: {round(performance['precision'], 4)}")
print(f"F1 Score: {round(performance['f1'], 4)}")
print(f"Recall: {round(performance['recall'], 4)}")

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM,0.966123,0.970858,accuracy,0.094155,0.015319,1.232771,0.094155,0.015319,1.232771,1,True,1
1,WeightedEnsemble_L2,0.966123,0.970858,accuracy,0.095975,0.016331,1.270120,0.001821,0.001012,0.037348,2,True,4
2,XGBoost,0.965314,0.970459,accuracy,0.072258,0.015225,2.021939,0.072258,0.015225,2.021939,1,True,3
3,CatBoost,0.964121,0.967665,accuracy,0.010697,0.003496,11.004228,0.010697,0.003496,11.004228,1,True,2


Desempenho do melhor modelo:
Acurácia: 0.9661
Precision: 0.9749
F1 Score: 0.9609
Recall: 0.9472
